<a href="https://colab.research.google.com/github/ayanrasulova/rent-estimations-cv/blob/main/finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/cv_project'
IMAGE_DIR = os.path.join(DRIVE_DIR, 'images')

print(f"Images available: {len(os.listdir(IMAGE_DIR))}")

Mounted at /content/drive
Images available: 8399


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [3]:
# load & clean data
df = pd.read_csv(os.path.join(DRIVE_DIR, 'listings_clean.csv'))
df['img_path'] = df['img_path'].apply(
    lambda p: os.path.join(IMAGE_DIR, os.path.basename(str(p))) if pd.notna(p) else None
)


# fix any remaining string columns
df['sqft'] = pd.to_numeric(df['sqft'].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price', 'img_path'])
df = df[df['img_path'].apply(os.path.exists)]

print(f"Listings with valid images: {len(df)}")
print(f"Price range: ${df['price'].min():,.0f} – ${df['price'].max():,.0f}")



Listings with valid images: 8400
Price range: $150,000 – $11,495,000


In [4]:
# dataset
class PropertyDataset(Dataset):
    def __init__(self, img_paths, prices, transform):
        self.img_paths = img_paths
        self.prices    = prices
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.img_paths[idx]).convert('RGB')
            img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        price = torch.tensor(self.prices[idx], dtype=torch.float32)
        return img, price

In [5]:
# transforms
# add augmentation for training to help generalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [6]:
# prep data
df_valid = df[df['img_path'].notna() & df['img_path'].apply(os.path.exists)].copy()
df_valid['price'] = pd.to_numeric(df_valid['price'], errors='coerce')
df_valid = df_valid.dropna(subset=['price'])

# normalize prices to help training (predict in $100k units)
price_mean = df_valid['price'].mean()
price_std  = df_valid['price'].std()
df_valid['price_norm'] = (df_valid['price'] - price_mean) / price_std

# train/val split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df_valid, test_size=0.2, random_state=42)

train_dataset = PropertyDataset(train_df['img_path'].tolist(), train_df['price_norm'].tolist(), train_transform)
val_dataset   = PropertyDataset(val_df['img_path'].tolist(),   val_df['price_norm'].tolist(),   val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# build model
model = models.resnet50(weights='IMAGENET1K_V1')

# freeze all layer
for param in model.parameters():
    param.requires_grad = False

# unfreeze the last ResNet block (layer4) + final FC
for param in model.layer4.parameters():
    param.requires_grad = True

# replace final classification layer with regression head
model.fc = nn.Sequential(
    nn.Linear(2048, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 1)   # single output = price
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"Using device: {device}")

Train: 6720 | Val: 1680
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]


Using device: cuda


In [7]:
# training loop
# only optimize unfrozen parameters
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
criterion = nn.MSELoss()

In [8]:
# training loop
EPOCHS = 10

for epoch in range(EPOCHS):
    # train
    model.train()
    train_loss = 0
    for imgs, prices in train_loader:
        imgs, prices = imgs.to(device), prices.to(device)
        optimizer.zero_grad()
        preds = model(imgs).squeeze(1)
        loss  = criterion(preds, prices)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, prices in val_loader:
            imgs, prices = imgs.to(device), prices.to(device)
            preds = model(imgs).squeeze(1)
            val_loss += criterion(preds, prices).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)

    # convert loss back to dollars (denormalize)
    train_mae_dollars = (train_loss ** 0.5) * price_std
    val_mae_dollars   = (val_loss   ** 0.5) * price_std

    scheduler.step()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ~Val $RMSE: ${val_mae_dollars:,.0f}")

Epoch 01/10 | Train Loss: 0.8562 | Val Loss: 0.7240 | ~Val $RMSE: $1,132,618
Epoch 02/10 | Train Loss: 0.5993 | Val Loss: 0.7671 | ~Val $RMSE: $1,165,806
Epoch 03/10 | Train Loss: 0.4076 | Val Loss: 0.8597 | ~Val $RMSE: $1,234,236
Epoch 04/10 | Train Loss: 0.2641 | Val Loss: 0.7419 | ~Val $RMSE: $1,146,544
Epoch 05/10 | Train Loss: 0.2006 | Val Loss: 0.6883 | ~Val $RMSE: $1,104,315
Epoch 06/10 | Train Loss: 0.1556 | Val Loss: 0.6944 | ~Val $RMSE: $1,109,225
Epoch 07/10 | Train Loss: 0.1314 | Val Loss: 0.6525 | ~Val $RMSE: $1,075,246
Epoch 08/10 | Train Loss: 0.1088 | Val Loss: 0.6602 | ~Val $RMSE: $1,081,566
Epoch 09/10 | Train Loss: 0.0954 | Val Loss: 0.6630 | ~Val $RMSE: $1,083,863
Epoch 10/10 | Train Loss: 0.0817 | Val Loss: 0.6810 | ~Val $RMSE: $1,098,466


In [9]:
# continue training
EPOCHS = 10  # 10 more epochs
for epoch in range(EPOCHS):
    # train
    model.train()
    train_loss = 0
    for imgs, prices in train_loader:
        imgs, prices = imgs.to(device), prices.to(device)
        optimizer.zero_grad()
        preds = model(imgs).squeeze(1)
        loss  = criterion(preds, prices)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, prices in val_loader:
            imgs, prices = imgs.to(device), prices.to(device)
            preds = model(imgs).squeeze(1)
            val_loss += criterion(preds, prices).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)

    # convert loss back to dollars (denormalize)
    train_mae_dollars = (train_loss ** 0.5) * price_std
    val_mae_dollars   = (val_loss   ** 0.5) * price_std

    scheduler.step()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ~Val $RMSE: ${val_mae_dollars:,.0f}")


Epoch 01/10 | Train Loss: 0.0767 | Val Loss: 0.6824 | ~Val $RMSE: $1,099,556
Epoch 02/10 | Train Loss: 0.0705 | Val Loss: 0.6701 | ~Val $RMSE: $1,089,613
Epoch 03/10 | Train Loss: 0.0614 | Val Loss: 0.6714 | ~Val $RMSE: $1,090,709
Epoch 04/10 | Train Loss: 0.0626 | Val Loss: 0.6646 | ~Val $RMSE: $1,085,159
Epoch 05/10 | Train Loss: 0.0624 | Val Loss: 0.6641 | ~Val $RMSE: $1,084,715
Epoch 06/10 | Train Loss: 0.0576 | Val Loss: 0.6648 | ~Val $RMSE: $1,085,354
Epoch 07/10 | Train Loss: 0.0543 | Val Loss: 0.6616 | ~Val $RMSE: $1,082,705
Epoch 08/10 | Train Loss: 0.0580 | Val Loss: 0.6670 | ~Val $RMSE: $1,087,104
Epoch 09/10 | Train Loss: 0.0481 | Val Loss: 0.6708 | ~Val $RMSE: $1,090,225
Epoch 10/10 | Train Loss: 0.0495 | Val Loss: 0.6679 | ~Val $RMSE: $1,087,887


In [10]:
# save to drive
torch.save(model.state_dict(), os.path.join(DRIVE_DIR, 'resnet50_finetuned.pth'))
print("Saved to Drive!")

Saved to Drive!


In [12]:

# extract features
model.eval()
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor = feature_extractor.to(device)

extract_dataset = PropertyDataset(df_valid['img_path'].tolist(), df_valid['price_norm'].tolist(), val_transform)
extract_loader  = DataLoader(extract_dataset, batch_size=32, shuffle=False, num_workers=2)

all_features = []
with torch.no_grad():
    for imgs, _ in extract_loader:
        imgs     = imgs.to(device)
        features = feature_extractor(imgs).squeeze(-1).squeeze(-1)
        all_features.append(features.cpu().numpy())

visual_features_ft = np.vstack(all_features)
np.save(os.path.join(DRIVE_DIR, 'visual_features_finetuned.npy'), visual_features_ft)
print(f"Fine-tuned features shape: {visual_features_ft.shape}")

Fine-tuned features shape: (8400, 2048)


In [14]:
from sklearn.decomposition import PCA

# reduce 2048 CNN dims down to 100
pca = PCA(n_components=100)
visual_features_pca = pca.fit_transform(visual_features_ft)
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

X_tab      = df_model[TABULAR].apply(pd.to_numeric, errors='coerce').values
X_combined = np.hstack([X_tab, visual_features_pca])  # much smaller now
y          = df['price'].values

imputer    = SimpleImputer(strategy='median')
X_combined = imputer.fit_transform(X_combined)

X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

models_cv = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
}

print(f"\n{'Model':20s} | {'MAE':>12} | {'RMSE':>12} | {'R²':>6}")
print("-" * 60)
for name, m in models_cv.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    print(f"{name:20s} | ${mae:>10,.0f} | ${rmse:>10,.0f} | {r2:.3f}")

Variance explained: 86.06%

Model                |          MAE |         RMSE |     R²
------------------------------------------------------------
Linear Regression    | $   571,239 | $   998,936 | 0.414
Decision Tree        | $   569,397 | $ 1,080,599 | 0.315
Random Forest        | $   551,966 | $ 1,064,144 | 0.335
